# 72. The Capacitated VRP (CVRP)

## Tier 2: The Classic Heuristic (Priority-Based Construction)

### Key assumptions
- Priority-based customer selection using multiple criteria
- Sequential vehicle filling until capacity constraints reached
- Composite scoring considering distance, demand, and urgency
- Real-time performance for practical problem sizes
- No backtracking - once assigned, customer stays with vehicle

### Approach (step-by-step)
1. **Priority Calculation**: Compute composite scores for all unassigned customers
2. **Customer Selection**: Select highest priority customer for current vehicle
3. **Capacity Check**: Verify vehicle capacity constraints
4. **Vehicle Assignment**: Assign customer to current vehicle or move to next vehicle
5. **Route Construction**: Build routes using nearest neighbor heuristic
6. **Performance Analysis**: Compare with mathematical optimization baseline

### What to look for in the results
- Customer priority rankings and selection order
- Vehicle capacity utilization patterns
- Total distance vs optimal solution comparison
- Computational efficiency and scalability
- Route quality and feasibility verification

### Concrete example (from the source)
For our 5-customer instance, the priority heuristic produces routes:
Vehicle 1: [2, 0], Vehicle 2: [3, 4], Vehicle 3: [1], with total distance 156.3 units.
This solution serves all customers while respecting capacity constraints and provides a good starting point for further optimization.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for heuristic implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import heapq
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully for priority-based CVRP heuristic")

Libraries imported successfully for priority-based CVRP heuristic


In [ ]:
# Define customer data structure for priority calculation
@dataclass
class Customer:
    """Customer data with priority calculation capabilities"""
    id: int
    x: float
    y: float
    demand: int
    distance_from_depot: float = 0.0
    priority_score: float = 0.0
    
    def calculate_distance_from_depot(self, depot_x: float, depot_y: float):
        """Calculate Euclidean distance from depot"""
        self.distance_from_depot = np.sqrt((self.x - depot_x)**2 + (self.y - depot_y)**2)
    
    def calculate_priority_score(self, weights: Dict[str, float] = None):
        """Calculate composite priority score based on multiple criteria"""
        if weights is None:
            # Default weights: prioritize closer customers with higher demand
            weights = {
                'distance': 0.4,      # Lower distance = higher priority
                'demand': 0.3,       # Higher demand = higher priority  
                'urgency': 0.3        # Customer ID as proxy for urgency
            }
        
        # Normalize components (lower distance = higher score, so invert)
        distance_score = 1.0 / (1.0 + self.distance_from_depot)
        demand_score = self.demand / 10.0  # Normalize by max expected demand
        urgency_score = (10 - self.id) / 10.0  # Lower ID = higher urgency
        
        # Calculate weighted composite score
        self.priority_score = (
            weights['distance'] * distance_score +
            weights['demand'] * demand_score +
            weights['urgency'] * urgency_score
        )
        
        return self.priority_score

print("Customer data class defined with priority scoring")

Customer data class defined with priority scoring


In [ ]:
# Priority-based CVRP construction heuristic
class PriorityCVRPHeuristic:
    """Priority-based construction heuristic for CVRP"""
    
    def __init__(self, depot_location: Tuple[float, float], vehicle_capacity: int):
        self.depot_location = depot_location
        self.vehicle_capacity = vehicle_capacity
        self.customers = []
        self.routes = {}
        self.total_distance = 0.0
        
    def add_customer(self, customer_id: int, x: float, y: float, demand: int):
        """Add a customer to the problem instance"""
        customer = Customer(customer_id, x, y, demand)
        customer.calculate_distance_from_depot(self.depot_location[0], self.depot_location[1])
        self.customers.append(customer)
    
    def solve(self, max_vehicles: int = 10, priority_weights: Dict[str, float] = None):
        """Solve CVRP using priority-based construction heuristic"""
        
        # Calculate priority scores for all customers
        for customer in self.customers:
            customer.calculate_priority_score(priority_weights)
        
        # Sort customers by priority (highest first)
        sorted_customers = sorted(self.customers, key=lambda c: c.priority_score, reverse=True)
        
        # Initialize routes
        self.routes = {}
        current_vehicle = 1
        current_route = []
        current_load = 0
        unassigned_customers = sorted_customers.copy()
        
        print(f"Starting priority-based assignment for {len(self.customers)} customers...")
        print(f"Customer priority order: {[c.id for c in sorted_customers]}")
        
        # Assign customers to vehicles
        while unassigned_customers and current_vehicle <= max_vehicles:
            customer_assigned = False
            
            # Try to assign the highest priority customer that fits
            for customer in unassigned_customers[:]:  # Copy to avoid modification during iteration
                if current_load + customer.demand <= self.vehicle_capacity:
                    # Assign customer to current vehicle
                    current_route.append(customer)
                    current_load += customer.demand
                    unassigned_customers.remove(customer)
                    
                    print(f"  Vehicle {current_vehicle}: Assigned Customer {customer.id} (demand={customer.demand}, priority={customer.priority_score:.3f})")
                    print(f"    Current load: {current_load}/{self.vehicle_capacity}")
                    customer_assigned = True
                    break
            
            # If no customer could be assigned, move to next vehicle
            if not customer_assigned:
                if current_route:  # If current vehicle has customers, finalize it
                    self.routes[current_vehicle] = current_route.copy()
                    print(f"  Vehicle {current_vehicle} finalized with {len(current_route)} customers (load: {current_load})")
                
                current_vehicle += 1
                current_route = []
                current_load = 0
                
                if current_vehicle <= max_vehicles:
                    print(f"  Moving to Vehicle {current_vehicle}")
        
        # Don't forget the last vehicle
        if current_route and current_vehicle <= max_vehicles:
            self.routes[current_vehicle] = current_route.copy()
            print(f"  Vehicle {current_vehicle} finalized with {len(current_route)} customers (load: {current_load})")
        
        # Build optimal routes within each vehicle using nearest neighbor
        self._build_vehicle_routes()
        
        # Calculate total distance
        self._calculate_total_distance()
        
        return self.routes
    
    def _build_vehicle_routes(self):
        """Build optimal routes within each vehicle using nearest neighbor heuristic"""
        
        for vehicle_id, customers in self.routes.items():
            if not customers:
                self.routes[vehicle_id] = []  # Empty route
                continue
            
            # Build route starting from depot
            route = [self.depot_location]
            unvisited = customers.copy()
            current_pos = self.depot_location
            
            while unvisited:
                # Find nearest unvisited customer
                nearest_customer = None
                nearest_distance = float('inf')
                
                for customer in unvisited:
                    dist = np.sqrt(
                        (current_pos[0] - customer.x)**2 + 
                        (current_pos[1] - customer.y)**2
                    )
                    if dist < nearest_distance:
                        nearest_distance = dist
                        nearest_customer = customer
                
                if nearest_customer:
                    route.append((nearest_customer.x, nearest_customer.y))
                    current_pos = (nearest_customer.x, nearest_customer.y)
                    unvisited.remove(nearest_customer)
            
            # Return to depot
            route.append(self.depot_location)
            
            # Store route as list of coordinates
            self.routes[vehicle_id] = route
    
    def _calculate_total_distance(self):
        """Calculate total distance traveled by all vehicles"""
        self.total_distance = 0.0
        
        for vehicle_id, route in self.routes.items():
            if len(route) < 2:
                continue
                
            route_distance = 0.0
            for i in range(len(route) - 1):
                dist = np.sqrt(
                    (route[i][0] - route[i+1][0])**2 + 
                    (route[i][1] - route[i+1][1])**2
                )
                route_distance += dist
            
            self.total_distance += route_distance
    
    def get_solution_summary(self) -> Dict:
        """Get comprehensive solution summary"""
        total_customers = sum(1 for route in self.routes.values() if len(route) > 2)
        total_demand = sum(
            customer.demand 
            for customers in self.routes.values() 
            if isinstance(customers, list) and customers and isinstance(customers[0], Customer)
            for customer in customers
        )
        
        vehicles_used = len([r for r in self.routes.values() if len(r) > 2])
        
        return {
            'total_distance': self.total_distance,
            'vehicles_used': vehicles_used,
            'total_customers': total_customers,
            'total_demand': total_demand,
            'routes': self.routes
        }

print("Priority-based CVRP heuristic class defined")

Priority-based CVRP heuristic class defined


In [ ]:
# Create the concrete example from the source material
# Using the same 5-customer instance as Tier 1
customer_data = [
    (1, 5, 10, 2),   # Customer 1: (x=5, y=10, demand=2)
    (2, 15, 5, 3),   # Customer 2: (x=15, y=5, demand=3)
    (3, 10, 15, 1),  # Customer 3: (x=10, y=15, demand=1)
    (4, 20, 10, 4),  # Customer 4: (x=20, y=10, demand=4)
    (5, 8, 12, 2)    # Customer 5: (x=8, y=12, demand=2)
]

# Create heuristic instance
heuristic = PriorityCVRPHeuristic(
    depot_location=(0, 0),
    vehicle_capacity=6
)

# Add customers
for customer_id, x, y, demand in customer_data:
    heuristic.add_customer(customer_id, x, y, demand)

print("CVRP instance created for priority-based heuristic:")
print(f"- Depot: {heuristic.depot_location}")
print(f"- Vehicle Capacity: {heuristic.vehicle_capacity}")
print(f"- Customers: {len(heuristic.customers)}")
print(f"- Total Demand: {sum(c.demand for c in heuristic.customers)}")

# Display customer priorities
print("\nCustomer Information:")
print(f"{'ID':<4} {'Location':<12} {'Demand':<8} {'Distance':<10} {'Priority':<10}")
print("-" * 50)
for customer in heuristic.customers:
    print(f"{customer.id:<4} ({customer.x},{customer.y}){customer.demand:<8} {customer.distance_from_depot:<10.2f} {customer.priority_score:<10.3f}")

CVRP instance created for priority-based heuristic:
- Depot: (0, 0)
- Vehicle Capacity: 6
- Customers: 5
- Total Demand: 12

Customer Information:
ID   Location     Demand   Distance   Priority  
--------------------------------------------------
1    (5,10)2        11.18      0.000     
2    (15,5)3        15.81      0.000     
3    (10,15)1        18.03      0.000     
4    (20,10)4        22.36      0.000     
5    (8,12)2        14.42      0.000     


In [ ]:
# Solve the CVRP using priority-based heuristic
start_time = time.time()

# Custom priority weights to emphasize different aspects
priority_weights = {
    'distance': 0.5,      # Higher weight on distance from depot
    'demand': 0.3,       # Moderate weight on demand
    'urgency': 0.2       # Lower weight on urgency
}

routes = heuristic.solve(max_vehicles=3, priority_weights=priority_weights)

end_time = time.time()
computation_time = end_time - start_time

print(f"\nHeuristic solution completed in {computation_time:.4f} seconds")
print(f"Total distance: {heuristic.total_distance:.2f}")

Starting priority-based assignment for 5 customers...
Customer priority order: [1, 2, 4, 3, 5]
  Vehicle 1: Assigned Customer 1 (demand=2, priority=0.281)
    Current load: 2/6
  Vehicle 1: Assigned Customer 2 (demand=3, priority=0.280)
    Current load: 5/6
  Vehicle 1: Assigned Customer 3 (demand=1, priority=0.196)
    Current load: 6/6
  Vehicle 1 finalized with 3 customers (load: 6)
  Moving to Vehicle 2
  Vehicle 2: Assigned Customer 4 (demand=4, priority=0.261)
    Current load: 4/6
  Vehicle 2: Assigned Customer 5 (demand=2, priority=0.192)
    Current load: 6/6
  Vehicle 2 finalized with 2 customers (load: 6)

Heuristic solution completed in 0.0001 seconds
Total distance: 94.19


In [ ]:
# Analyze and display the solution
def analyze_heuristic_solution(heuristic):
    """Comprehensive analysis of the heuristic solution"""
    
    print("\n" + "="*60)
    print("PRIORITY-BASED CVRP HEURISTIC SOLUTION ANALYSIS")
    print("="*60)
    
    # Solution summary
    summary = heuristic.get_solution_summary()
    
    print(f"\nOverall Performance:")
    print(f"  • Total Distance: {summary['total_distance']:.2f} units")
    print(f"  • Vehicles Used: {summary['vehicles_used']}")
    print(f"  • Customers Served: {summary['total_customers']}")
    print(f"  • Total Demand Served: {summary['total_demand']}")
    print(f"  • Computation Time: {computation_time:.4f} seconds")
    
    print(f"\nDetailed Route Analysis:")
    print(f"{'Vehicle':<8} {'Route':<30} {'Distance':<10} {'Customers':<12} {'Load':<8} {'Util%':<8}")
    print("-" * 80)
    
    for vehicle_id, route in routes.items():
        if len(route) <= 2:  # Empty route (depot to depot)
            continue
            
        # Calculate route distance
        route_distance = 0.0
        for i in range(len(route) - 1):
            dist = np.sqrt(
                (route[i][0] - route[i+1][0])**2 + 
                (route[i][1] - route[i+1][1])**2
            )
            route_distance += dist
        
        # Find customers in this route
        route_customers = []
        route_load = 0
        
        for customer in heuristic.customers:
            if (customer.x, customer.y) in route[1:-1]:  # Exclude depot at start and end
                route_customers.append(customer.id)
                route_load += customer.demand
        
        utilization = (route_load / heuristic.vehicle_capacity) * 100
        
        # Format route for display
        route_str = "Depot"
        for customer_id in route_customers:
            route_str += f"→C{customer_id}"
        route_str += "→Depot"
        
        print(f"{vehicle_id:<8} {route_str:<30} {route_distance:<10.2f} {len(route_customers):<12} {route_load:<8} {utilization:<8.1f}")
    
    return summary

# Analyze the solution
solution_summary = analyze_heuristic_solution(heuristic)


PRIORITY-BASED CVRP HEURISTIC SOLUTION ANALYSIS

Overall Performance:
  • Total Distance: 94.19 units
  • Vehicles Used: 2
  • Customers Served: 2
  • Total Demand Served: 0
  • Computation Time: 0.0001 seconds

Detailed Route Analysis:
Vehicle  Route                          Distance   Customers    Load     Util%   
--------------------------------------------------------------------------------
1        Depot→C1→C2→C3→Depot           45.24      3            6        100.0   
2        Depot→C4→C5→Depot              48.95      2            6        100.0   


In [ ]:
# Visualization of priority-based heuristic solution
def visualize_heuristic_solution(heuristic):
    """Create comprehensive visualization of the heuristic solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # Color palette for vehicles
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    # Plot 1: Route visualization
    ax1.set_title('Priority-Based Heuristic Routes', fontsize=14, fontweight='bold')
    
    # Plot depot
    ax1.plot(heuristic.depot_location[0], heuristic.depot_location[1], 
            'ks', markersize=15, label='Depot', zorder=5)
    
    # Plot customers with priority information
    for customer in heuristic.customers:
        ax1.plot(customer.x, customer.y, 'o', markersize=8, color='black', zorder=4)
        ax1.annotate(f'C{customer.id}\n(d={customer.demand}\np={customer.priority_score:.2f})', 
                    (customer.x, customer.y), xytext=(5, 5), 
                    textcoords='offset points', fontsize=8, zorder=6)
    
    # Plot routes
    for vehicle_id, route in routes.items():
        if len(route) <= 2:  # Skip empty routes
            continue
            
        color = colors[(vehicle_id-1) % len(colors)]
        
        # Plot route path
        for i in range(len(route) - 1):
            x_coords = [route[i][0], route[i+1][0]]
            y_coords = [route[i][1], route[i+1][1]]
            ax1.plot(x_coords, y_coords, color=color, linewidth=2, 
                    alpha=0.7, label=f'Vehicle {vehicle_id}', zorder=3)
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Customer priority distribution
    ax2.set_title('Customer Priority Distribution', fontsize=14, fontweight='bold')
    
    customer_ids = [c.id for c in heuristic.customers]
    priorities = [c.priority_score for c in heuristic.customers]
    demands = [c.demand for c in heuristic.customers]
    distances = [c.distance_from_depot for c in heuristic.customers]
    
    x_pos = np.arange(len(customer_ids))
    width = 0.25
    
    ax2.bar(x_pos - width, priorities, width, label='Priority Score', alpha=0.7, color='gold')
    ax2.bar(x_pos, [d/10 for d in demands], width, label='Demand/10', alpha=0.7, color='darkblue')
    ax2.bar(x_pos + width, [1/(1+d) for d in distances], width, label='1/(1+Distance)', alpha=0.7, color='darkgreen')
    
    ax2.set_xlabel('Customer ID')
    ax2.set_ylabel('Score')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(customer_ids)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Vehicle performance comparison
    ax3.set_title('Vehicle Performance Metrics', fontsize=14, fontweight='bold')
    
    vehicle_ids = []
    vehicle_distances = []
    vehicle_loads = []
    vehicle_customers = []
    
    for vehicle_id, route in routes.items():
        if len(route) <= 2:  # Skip empty routes
            continue
            
        # Calculate route distance
        route_distance = 0.0
        for i in range(len(route) - 1):
            dist = np.sqrt(
                (route[i][0] - route[i+1][0])**2 + 
                (route[i][1] - route[i+1][1])**2
            )
            route_distance += dist
        
        # Calculate load and customers
        route_load = 0
        route_customers = 0
        for customer in heuristic.customers:
            if (customer.x, customer.y) in route[1:-1]:
                route_load += customer.demand
                route_customers += 1
        
        vehicle_ids.append(vehicle_id)
        vehicle_distances.append(route_distance)
        vehicle_loads.append(route_load)
        vehicle_customers.append(route_customers)
    
    # Create grouped bar chart
    x_pos = np.arange(len(vehicle_ids))
    width = 0.25
    
    ax3.bar(x_pos - width, vehicle_distances, width, label='Distance', alpha=0.7, color='red')
    ax3.bar(x_pos, vehicle_loads, width, label='Load', alpha=0.7, color='blue')
    ax3.bar(x_pos + width, [c*5 for c in vehicle_customers], width, label='Customers×5', alpha=0.7, color='green')
    
    ax3.set_xlabel('Vehicle ID')
    ax3.set_ylabel('Value')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(vehicle_ids)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Algorithm performance summary
    ax4.set_title('Heuristic Performance Summary', fontsize=14, fontweight='bold')
    ax4.axis('off')
    
    # Create performance text
    performance_text = f"""
    Priority-Based Heuristic Results:
    ─────────────────────────────
    • Total Distance: {heuristic.total_distance:.2f} units
    • Vehicles Used: {len([r for r in routes.values() if len(r) > 2])}
    • Customers Served: {solution_summary['total_customers']}
    • Total Demand: {solution_summary['total_demand']}
    • Fleet Capacity: {len([r for r in routes.values() if len(r) > 2]) * heuristic.vehicle_capacity}
    • Capacity Utilization: {solution_summary['total_demand']/(len([r for r in routes.values() if len(r) > 2]) * heuristic.vehicle_capacity)*100:.1f}%
    • Computation Time: {computation_time:.4f} seconds
    • Algorithm Type: Priority-Based Construction
    • Solution Quality: Heuristic (not optimal)
    """
    
    ax4.text(0.1, 0.5, performance_text, fontsize=11, verticalalignment='center',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Visualize the heuristic solution
visualize_heuristic_solution(heuristic)

Iteration 150: Best Fitness = 20000.00, Diversity = 371.65
Iteration 199: Best Fitness = 20000.00, Diversity = 308.74

=== PSO Optimization Complete ===
Final best fitness: 20000.00
Computation time: 1.60 seconds
Convergence at iteration: 12
  Best cost: $20,000
  Convergence: 12 iterations

Testing Exploration parameters: w=0.9, c1=2.0, c2=1.0
Initializing PSO swarm with 30 particles...
Initial global best fitness: 53560.20
Initializing PSO swarm with 50 particles...
Initial global best fitness: 48058.86

=== Starting PSO Optimization ===
Swarm size: 50
Max iterations: 200
Problem dimension: 9
Iteration   0: Best Fitness = 31267.41, Diversity = 277.30
Iteration  50: Best Fitness = 20000.00, Diversity = 497.21
Iteration 100: Best Fitness = 20000.00, Diversity = 430.04


In [ ]:
# Performance comparison with different priority weight configurations
def compare_priority_weights():
    """Compare heuristic performance with different priority weight configurations"""
    
    weight_configs = [
        {'name': 'Distance Focused', 'weights': {'distance': 0.7, 'demand': 0.2, 'urgency': 0.1}},
        {'name': 'Demand Focused', 'weights': {'distance': 0.2, 'demand': 0.7, 'urgency': 0.1}},
        {'name': 'Balanced', 'weights': {'distance': 0.33, 'demand': 0.33, 'urgency': 0.34}},
        {'name': 'Urgency Focused', 'weights': {'distance': 0.2, 'demand': 0.2, 'urgency': 0.6}}
    ]
    
    results = []
    
    print("\nPriority Weight Configuration Comparison:")
    print("=" * 60)
    print(f"{'Configuration':<15} {'Distance':<10} {'Vehicles':<10} {'Time (s)':<10} {'Load Util%':<10}")
    print("-" * 60)
    
    for config in weight_configs:
        # Create new heuristic instance
        h = PriorityCVRPHeuristic((0, 0), 6)
        
        # Add customers
        for customer_id, x, y, demand in customer_data:
            h.add_customer(customer_id, x, y, demand)
        
        # Solve with specific weights
        start_time = time.time()
        routes = h.solve(max_vehicles=3, priority_weights=config['weights'])
        end_time = time.time()
        
        # Calculate metrics
        vehicles_used = len([r for r in routes.values() if len(r) > 2])
        total_demand = sum(c.demand for c in h.customers)
        total_capacity = vehicles_used * h.vehicle_capacity
        utilization = (total_demand / total_capacity * 100) if total_capacity > 0 else 0
        
        results.append({
            'config': config['name'],
            'distance': h.total_distance,
            'vehicles': vehicles_used,
            'time': end_time - start_time,
            'utilization': utilization
        })
        
        print(f"{config['name']:<15} {h.total_distance:<10.2f} {vehicles_used:<10} {end_time-start_time:<10.4f} {utilization:<10.1f}")
    
    return results

# Compare different weight configurations
weight_comparison_results = compare_priority_weights()


Priority Weight Configuration Comparison:
Configuration   Distance   Vehicles   Time (s)   Load Util%
------------------------------------------------------------
Starting priority-based assignment for 5 customers...
Customer priority order: [1, 2, 4, 5, 3]
  Vehicle 1: Assigned Customer 1 (demand=2, priority=0.187)
    Current load: 2/6
  Vehicle 1: Assigned Customer 2 (demand=3, priority=0.182)
    Current load: 5/6
  Vehicle 1: Assigned Customer 3 (demand=1, priority=0.127)
    Current load: 6/6
  Vehicle 1 finalized with 3 customers (load: 6)
  Moving to Vehicle 2
  Vehicle 2: Assigned Customer 4 (demand=4, priority=0.170)
    Current load: 4/6
  Vehicle 2: Assigned Customer 5 (demand=2, priority=0.135)
    Current load: 6/6
  Vehicle 2 finalized with 2 customers (load: 6)
Distance Focused 94.19      2          0.0001     100.0     
Starting priority-based assignment for 5 customers...
Customer priority order: [4, 2, 1, 5, 3]
  Vehicle 1: Assigned Customer 4 (demand=4, priority=0.

In [ ]:
# Scalability analysis with larger problem instances
def scalability_analysis():
    """Test heuristic scalability with increasing problem sizes"""
    
    problem_sizes = [5, 10, 15, 20, 25]
    results = []
    
    print("\nScalability Analysis:")
    print("=" * 50)
    print(f"{'Customers':<10} {'Vehicles':<10} {'Distance':<10} {'Time (s)':<10} {'Quality':<10}")
    print("-" * 50)
    
    for size in problem_sizes:
        # Generate random problem instance
        np.random.seed(42 + size)  # Different seed for each size
        
        # Create heuristic instance
        h = PriorityCVRPHeuristic((0, 0), 10)  # Larger capacity for larger problems
        
        # Generate random customers
        for i in range(1, size + 1):
            x = np.random.uniform(0, 30)
            y = np.random.uniform(0, 30)
            demand = np.random.randint(1, 5)
            h.add_customer(i, x, y, demand)
        
        # Solve
        start_time = time.time()
        routes = h.solve(max_vehicles=min(10, size // 2 + 1))
        end_time = time.time()
        
        # Calculate quality metric (distance per customer)
        quality_metric = h.total_distance / size if size > 0 else 0
        
        vehicles_used = len([r for r in routes.values() if len(r) > 2])
        
        results.append({
            'size': size,
            'vehicles': vehicles_used,
            'distance': h.total_distance,
            'time': end_time - start_time,
            'quality': quality_metric
        })
        
        print(f"{size:<10} {vehicles_used:<10} {h.total_distance:<10.2f} {end_time-start_time:<10.4f} {quality_metric:<10.2f}")
    
    return results

# Run scalability analysis
scalability_results = scalability_analysis()


Scalability Analysis:
Customers  Vehicles   Distance   Time (s)   Quality   
--------------------------------------------------
Starting priority-based assignment for 5 customers...
Customer priority order: [1, 2, 3, 4, 5]
  Vehicle 1: Assigned Customer 1 (demand=2, priority=0.363)
    Current load: 2/10
  Vehicle 1: Assigned Customer 2 (demand=2, priority=0.321)
    Current load: 4/10
  Vehicle 1: Assigned Customer 3 (demand=2, priority=0.287)
    Current load: 6/10
  Vehicle 1: Assigned Customer 4 (demand=1, priority=0.230)
    Current load: 7/10
  Vehicle 1: Assigned Customer 5 (demand=2, priority=0.227)
    Current load: 9/10
  Vehicle 1 finalized with 5 customers (load: 9)
5          1          55.31      0.0001     11.06     
Convergence metrics:
  Generations to reach target: 1
  Early improvement (20 gen): 66.5%
  Final improvement: 66.5%
  Convergence efficiency: 66.51% per generation


### Why this Tier exists vs Tier 1
The priority-based construction heuristic addresses key limitations of the mathematical optimization approach:

**Computational Efficiency:**
- Solves problems in milliseconds vs minutes/hours for MIP
- Scales to much larger problem instances (100+ customers)
- Suitable for real-time decision making

**Practical Applicability:**
- Handles dynamic environments with frequent changes
- Provides good-enough solutions quickly
- Easy to implement and modify for specific constraints

**Algorithmic Intuition:**
- Mimics human decision-making processes
- Transparent and explainable solution process
- Allows incorporation of domain knowledge through priority weights

### Pros / Cons vs Tier 1

**Pros:**
- **Speed**: Orders of magnitude faster than MIP
- **Scalability**: Handles much larger problem instances
- **Simplicity**: Easy to understand and implement
- **Flexibility**: Priority weights can be tuned for specific scenarios
- **Robustness**: Less sensitive to problem formulation details

**Cons:**
- **Optimality Gap**: No guarantee of optimal solution
- **Solution Quality**: May be significantly worse than optimal
- **Local Optima**: Can get stuck in poor local solutions
- **Parameter Sensitivity**: Performance depends on weight selection
- **Limited Search**: No systematic exploration of solution space

### When to use this Tier
- **Real-time routing** where quick decisions are essential
- **Large-scale problems** where MIP is computationally infeasible
- **Dynamic environments** with frequent parameter changes
- **Initial solution generation** for local search improvement
- **Resource-constrained environments** with limited computational power
- **Benchmark comparison** to evaluate more sophisticated methods